# 01. 데이터 파싱 및 전처리 (Data Processing)
이 노트북은 원본 데이터에서 숭실대 인근 통학거리와 월세를 분석하기 위한 최적의 데이터셋을 구축합니다.

### 전처리 기준
1. **숭실대 인근 6개 동 타겟팅**
2. **보증금 1억 원 이하, 월세 100만 원 이하 필터링**
3. **'연립다세대' 및 '오피스텔' 추출**: 본번/부번을 통한 정확한 주소 지오코딩이 불가능한 '단독다가구' 제외
4. **숭실대 통학 가능 직선거리 계산 (1200m 이하 선별)**


In [1]:
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

target_dongs = ["상도1동", "상도동", "사당동", "흑석동", "봉천동", "대방동"]
docs_dir = "data/original"
files = [f for f in os.listdir(docs_dir) if f.startswith("서울특별시_전월세가_") and f.endswith(".csv")]

df_list = []
for file in sorted(files):
    file_path = os.path.join(docs_dir, file)
    print(f"[{file}] 로드 중...")
    try:
        df = pd.read_csv(file_path, encoding='cp949')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='utf-8')
    
    # 전월세 중 '월세' 거래 건만 추출
    df = df[df['전월세구분'] == '월세']
    
    # 숭실대 인근 동 필터링
    if '법정동명' in df.columns:
        df_filtered = df[df['법정동명'].isin(target_dongs)].copy()
    else:
        continue
    
    # 대학생 현실 기준: 보증금 1억 이하, 월세 100만 이하 매물 추출
    df_filtered = df_filtered[(df_filtered['보증금(만원)'] <= 10000) & (df_filtered['임대료(만원)'] <= 100)]
    
    # 건물용도 필터링 (단독다가구 제외)
    df_filtered = df_filtered[df_filtered['건물용도'].isin(['연립다세대', '오피스텔'])]
    
    # 본번/부번이 존재하는 데이터만 추출
    df_filtered = df_filtered[df_filtered['본번'].notnull() & df_filtered['부번'].notnull()]
    
    df_list.append(df_filtered)

raw_df = pd.concat(df_list, ignore_index=True)
print(f"\n✅ 1차 파싱 완료: 총 {len(raw_df):,}건 추출")


[서울특별시_전월세가_2023.csv] 로드 중...
[서울특별시_전월세가_2024.csv] 로드 중...
[서울특별시_전월세가_2025.csv] 로드 중...

✅ 1차 파싱 완료: 총 15,011건 추출


In [2]:
# 2. 도로명/지번 주소 병합 및 통학 거리 산출
def make_address(row):
    addr = f"서울특별시 {row['자치구명']} {row['법정동명']}"
    bon = row.get('본번', None)
    bu = row.get('부번', None)
    if pd.notnull(bon) and str(bon).strip() != "":
        bon_str = str(int(bon)) if isinstance(bon, float) or isinstance(bon, int) else str(bon)
        addr += f" {bon_str}"
        if pd.notnull(bu) and str(bu).strip() != "" and float(bu) > 0:
            bu_str = str(int(bu)) if isinstance(bu, float) or isinstance(bu, int) else str(bu)
            addr += f"-{bu_str}"
    return addr

raw_df['주소'] = raw_df.apply(make_address, axis=1)

cache_path = "data/geocode_cache.csv"
if os.path.exists(cache_path):
    cache_df = pd.read_csv(cache_path)
    # 캐시와 병합하여 위도, 경도, 직선거리 획득
    merged_df = pd.merge(raw_df, cache_df[['주소', '위도', '경도', '직선거리(m)']].drop_duplicates(subset=['주소']), on="주소", how="inner")
else:
    print("Error: geocode_cache.csv 가 필요합니다.")

# 1200m 이내 통학거리 데이터만 최종 필터링
merged_df = merged_df[merged_df['직선거리(m)'] <= 1200]

# 월 실질주거비 파생변수 생성 (보증금/1000*5.5 + 월세)
merged_df["월실질주거비(만원)"] = merged_df["임대료(만원)"] + (merged_df["보증금(만원)"] / 1000 * 5.5)

# 분석에 불필요한 칼럼 제거 및 정리
final_cols = ["주소", "위도", "경도", "직선거리(m)", "임대면적", "보증금(만원)", "임대료(만원)", "건축년도", "월실질주거비(만원)"]
final_df = merged_df[final_cols].copy()

# 최종 데이터 저장
os.makedirs("data", exist_ok=True)
final_df.to_csv("data/final_rent_data.csv", index=False)

print(f"✅ 최종 전처리 완료: 통학 1200m 이내 총 {len(final_df):,}건 확보")
print("데이터셋이 'data/final_rent_data.csv'로 저장되었습니다.")
display(final_df.head())


✅ 최종 전처리 완료: 통학 1200m 이내 총 1,854건 확보
데이터셋이 'data/final_rent_data.csv'로 저장되었습니다.


,주소,위도,경도,직선거리(m),임대면적,보증금(만원),임대료(만원),건축년도,월실질주거비(만원)
37,서울특별시 관악구 봉천동 41-168,37.488118,126.953817,971.1,38.16,500,40,2001.0,42.75
42,서울특별시 동작구 상도1동 450-2,37.502439,126.950941,867.6,66.36,1000,35,2012.0,40.50
43,서울특별시 동작구 상도1동 450-2,37.502439,126.950941,867.6,66.36,1000,35,2012.0,40.50
48,서울특별시 동작구 상도동 466-1,37.499100,126.952315,524.2,23.57,1000,52,2003.0,57.50
61,서울특별시 동작구 상도동 495-2,37.494960,126.954383,297.8,14.26,500,45,1995.0,47.75
